# SmolVLM-256M · Revised protocol without MediaPipe

| | |
|---|---|
| **Project** | EdgeVLM-Drive — resource-aware vision-language driver monitoring on NVIDIA Jetson Nano |
| **Model** | SmolVLM-256M-Instruct |
| **Stage** | 9 of 12 · Revised protocol without MediaPipe |
| **Purpose** | Revised prompt, preprocessing and training settings; Haar-based face crops. |
| **Input** | train / validation / test CSVs with `frame_path` and `label` (see `data/README.md`) |
| **Output** | LoRA adapter (`best_by_macro_f1`), predictions CSV, metrics |
| **Configuration** | [`09_revised_no_mediapipe.json`](../../configs/smolvlm256m/09_revised_no_mediapipe.json) |
| **Results** | [`revised_no_mediapipe.json`](../../results/smolvlm_256m/revised_no_mediapipe.json) |
| **Type** | Training source; metrics are stored in the results file above |
| **Setup** | set the paths in the first configuration cell to your data and split folders |


# EdgeVLM-Drive — Improved SmolVLM-256M Phase 2 experiment

This is a new, isolated experiment; it does not overwrite earlier model outputs.
It keeps the subject-disjoint Phase 2 splits and imports the shared preprocessing
from `langgraph_pipeline.py`. It does **not** add MediaPipe.

Changes in this experiment:

1. A preprocessing audit reports face-detection, blur-quality, and face-crop texture checks by class, and saves examples for visual review. These are warnings, not automatic exclusions.
2. The prompt gives concise, evidence-based definitions for the four labels, especially PHONE_USAGE versus LOOKING_AWAY.
3. Training-only mild brightness/contrast augmentation is available for LOOKING_AWAY; validation and test images remain unchanged.
4. Class balancing is configurable (`sqrt`, `inverse`, or `none`). The recommended initial setting is the less aggressive `sqrt` sampler.
5. The default starts from the untouched pretrained base to avoid stacking a class-weighted adapter with another balancing method. The learning rate is reduced to 5e-5, and early stopping uses a larger balanced validation sample.
6. A new output directory prevents old checkpoints or predictions from being reused.

The test split remains untouched during model selection. Treat the final test metrics as the final report for the selected configuration, not as a tuning signal. For clean ablations, change one setting at a time and use a distinct output directory.


## 1. Configuration and production preprocessing


In [ ]:
import os
import sys
import importlib.util
from pathlib import Path

# Keep this experiment on one GPU. This avoids the DataParallel stall observed earlier.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

HOME = Path("/home/jovyan")
SPLIT_DIR = HOME / "phase2-tain-val-test-splits"
TRAIN_CSV = SPLIT_DIR / "final_train.csv"
VAL_CSV = SPLIT_DIR / "final_val.csv"
TEST_CSV = SPLIT_DIR / "final_test.csv"

PROJECT_ROOT = HOME / "EdgeVLM_finetune vlm256"
SAMPLER_MODE = "sqrt"  # options: "sqrt", "inverse", "none"
AUGMENT_LOOKING_AWAY_TRAIN = True  # mild brightness/contrast only; train split only
START_FROM_PREVIOUS_ADAPTER = False  # clean baseline; avoids stacking class balancing
LEARNING_RATE = 5e-5
MAX_EPOCHS = 6
EARLY_STOPPING_PATIENCE = 2
MACRO_F1_SAMPLES_PER_CLASS = 150
if SAMPLER_MODE not in {"sqrt", "inverse", "none"}:
    raise ValueError("SAMPLER_MODE must be sqrt, inverse, or none")
OUTPUT_ROOT = PROJECT_ROOT / (
    f"phase2_256m_improved_{SAMPLER_MODE}_"
    f"lookawayaug_{int(AUGMENT_LOOKING_AWAY_TRAIN)}_lr{LEARNING_RATE:g}"
)
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
LAST_ADAPTER_DIR = OUTPUT_ROOT / "last_adapter"
BEST_ADAPTER_DIR = OUTPUT_ROOT / "best_by_macro_f1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
MODEL_CACHE_CANDIDATES = [
    PROJECT_ROOT / "model_cache",
    HOME / "EdgeVLM_finetune" / "model_cache",
]
MODEL_CACHE = next((p for p in MODEL_CACHE_CANDIDATES if p.exists()), MODEL_CACHE_CANDIDATES[0])
MODEL_CACHE.mkdir(parents=True, exist_ok=True)

PREVIOUS_ADAPTER_CANDIDATES = [
    PROJECT_ROOT / "smolvlm_256m_lora_augmented_classweighted",
    HOME / "EdgeVLM_finetune_vlm256" / "smolvlm_256m_lora_augmented_classweighted",
    HOME / "EdgeVLM_finetune" / "smolvlm_256m_lora_augmented_classweighted",
]
PREVIOUS_ADAPTER_DIR = next(
    (p for p in PREVIOUS_ADAPTER_CANDIDATES if (p / "adapter_model.safetensors").is_file()),
    None,
)

LABELS = ["NORMAL", "DROWSY", "LOOKING_AWAY", "PHONE_USAGE"]
IMAGE_SIZE = 512
SEED = 42

LANGGRAPH_MODULE_PATH = HOME / "EdgeVLM_finetune" / "langgraph_pipeline.py"
if not LANGGRAPH_MODULE_PATH.is_file():
    raise FileNotFoundError(
        f"Production preprocessing module not found: {LANGGRAPH_MODULE_PATH}"
    )

spec = importlib.util.spec_from_file_location("langgraph_pipeline", LANGGRAPH_MODULE_PATH)
lgp = importlib.util.module_from_spec(spec)
sys.modules["langgraph_pipeline"] = lgp
spec.loader.exec_module(lgp)

print("LangGraph preprocessing:", lgp.__file__)
print("Full-frame size:", lgp.FULL_FRAME_SIZE)
print("Face-crop size:", lgp.FACE_CROP_SIZE)
print("Haar fallback enabled:", lgp.ENABLE_HAAR_FACE_FALLBACK)
print("Previous adapter:", PREVIOUS_ADAPTER_DIR)
print("Use previous adapter:", START_FROM_PREVIOUS_ADAPTER)
print("Output directory:", OUTPUT_ROOT)
for path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    print(path, "exists:", path.is_file())
    if not path.is_file():
        raise FileNotFoundError(path)


## 2. Read, validate, and deduplicate Phase 2 data


In [ ]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)
datasets = {"train": train_df, "validation": val_df, "test": test_df}
required_columns = {"frame_path", "label", "source_dataset", "subject"}


def resolve_image_path(value):
    path = Path(str(value).strip().replace("\\", "/"))
    return path if path.is_absolute() else HOME / path


for split_name, dataframe in datasets.items():
    missing_columns = required_columns - set(dataframe.columns)
    if missing_columns:
        raise ValueError(f"{split_name} missing columns: {sorted(missing_columns)}")
    dataframe["label"] = dataframe["label"].astype(str).str.strip().str.upper()
    unknown = sorted(set(dataframe["label"]) - set(LABELS))
    if unknown:
        raise ValueError(f"{split_name} contains unknown labels: {unknown}")
    dataframe["resolved_path"] = dataframe["frame_path"].map(
        lambda value: str(resolve_image_path(value))
    )
    missing = [p for p in dataframe["resolved_path"] if not Path(p).is_file()]
    print(f"\n{split_name}: {len(dataframe)} rows | missing images: {len(missing)}")
    print(dataframe["label"].value_counts().reindex(LABELS, fill_value=0))
    if missing:
        print("First missing path:", missing[0])

if any(not Path(p).is_file() for df in datasets.values() for p in df["resolved_path"]):
    raise FileNotFoundError("Fix missing image paths before training.")

for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    image_overlap = set(datasets[left]["resolved_path"]) & set(datasets[right]["resolved_path"])
    print(f"Exact image overlap {left}/{right}:", len(image_overlap))
    if image_overlap:
        raise ValueError(f"Image leakage detected between {left} and {right}")


def subject_keys(dataframe):
    return set(zip(dataframe["source_dataset"].astype(str), dataframe["subject"].astype(str)))


for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    overlap = subject_keys(datasets[left]) & subject_keys(datasets[right])
    print(f"Subject overlap {left}/{right}:", len(overlap))
    if overlap:
        raise ValueError(f"Subject leakage detected between {left} and {right}: {list(overlap)[:5]}")

before = len(train_df)
train_df = train_df.drop_duplicates(subset=["resolved_path"], keep="first").reset_index(drop=True)
datasets["train"] = train_df
print("\nTraining duplicates removed:", before - len(train_df))
print("Final training rows:", len(train_df))
print(train_df["label"].value_counts().reindex(LABELS, fill_value=0))


## 3. Verify dual-crop preprocessing on real images


In [ ]:
import cv2
import numpy as np
from PIL import Image


def create_dual_crops(image_path):
    raw_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if raw_bgr is None:
        raise FileNotFoundError(f"OpenCV could not read: {image_path}")
    enhanced_rgb, frame_quality_ok = lgp.enhance_frame(raw_bgr)
    full_crop = lgp.resize_full_frame(enhanced_rgb)
    face_crop = lgp.extract_face_crop(enhanced_rgb, landmarks=None)
    face_detected = lgp.detect_face_bbox_haar(enhanced_rgb) is not None
    return full_crop, face_crop, bool(frame_quality_ok), face_detected


RUN_PREPROCESSING_SANITY_CHECK = True
AUDIT_SAMPLES_PER_CLASS = 100
if RUN_PREPROCESSING_SANITY_CHECK:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    visual_rows = pd.concat([
        train_df[train_df["label"] == label].sample(n=min(2, (train_df["label"] == label).sum()), random_state=SEED)
        for label in LABELS
    ]).reset_index(drop=True)

    fig, axes = plt.subplots(len(visual_rows), 3, figsize=(10, 3 * len(visual_rows)), squeeze=False)
    for index, row in visual_rows.iterrows():
        raw_bgr = cv2.imread(str(row["resolved_path"]), cv2.IMREAD_COLOR)
        full_crop, face_crop, quality_ok, face_detected = create_dual_crops(row["resolved_path"])
        axes[index, 0].imshow(cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB))
        axes[index, 0].set_title(f"{row['label']} | original")
        axes[index, 1].imshow(full_crop)
        axes[index, 1].set_title("full available frame")
        axes[index, 2].imshow(face_crop)
        axes[index, 2].set_title(f"face crop | Haar={face_detected}")
        for ax in axes[index]:
            ax.axis("off")
    plt.tight_layout()
    sanity_path = OUTPUT_ROOT / "preprocessing_sanity_check.png"
    fig.savefig(sanity_path, dpi=110)
    plt.close(fig)
    print("Saved visual check:", sanity_path)

    audit_rows = []
    texture_threshold = float(getattr(lgp, "FACE_CROP_MIN_STD", 30.0))
    for label in LABELS:
        subset = train_df[train_df["label"] == label]
        sample = subset.sample(n=min(AUDIT_SAMPLES_PER_CLASS, len(subset)), random_state=SEED)
        for _, row in sample.iterrows():
            full_crop, face_crop, quality_ok, face_detected = create_dual_crops(row["resolved_path"])
            gray = cv2.cvtColor(face_crop, cv2.COLOR_RGB2GRAY)
            crop_std = float(np.std(gray))
            audit_rows.append({
                "label": label,
                "frame_path": row["frame_path"],
                "haar_face_detected": face_detected,
                "frame_quality_ok": quality_ok,
                "face_crop_gray_std": crop_std,
                "low_texture_warning": crop_std < texture_threshold,
                "review_warning": (not face_detected) or (not quality_ok) or (crop_std < texture_threshold),
            })

    quality_audit_df = pd.DataFrame(audit_rows)
    audit_path = OUTPUT_ROOT / "preprocessing_quality_audit.csv"
    quality_audit_df.to_csv(audit_path, index=False)
    print("Saved per-image preprocessing audit:", audit_path)
    print("Sampled training-set checks by class (warnings do not remove images):")
    print(quality_audit_df.groupby("label").agg(
        samples=("label", "size"),
        face_detected_pct=("haar_face_detected", lambda x: round(100 * x.mean(), 1)),
        blur_quality_ok_pct=("frame_quality_ok", lambda x: round(100 * x.mean(), 1)),
        low_texture_pct=("low_texture_warning", lambda x: round(100 * x.mean(), 1)),
        review_warning_pct=("review_warning", lambda x: round(100 * x.mean(), 1)),
    ).reindex(LABELS))
    print("Texture is only a crop-quality proxy; inspect the saved image grid to confirm the crop shows the driver.")
else:
    print("Preprocessing audit skipped.")


The `full_crop` is the complete **available input file**. If the dataset file is
already a face-only frame, this step cannot recreate missing hands or cabin context.
The saved sanity-check image must therefore be reviewed before trusting PHONE_USAGE
or LOOKING_AWAY performance.


## 4. Verify the existing offline training environment


In [ ]:
import torch
import transformers
import tokenizers
import peft
import accelerate
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import PeftModel, LoraConfig, TaskType, get_peft_model

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this notebook.")
print("Visible GPUs:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("OpenCV:", cv2.__version__)

if transformers.__version__ != "4.46.3":
    raise RuntimeError("Use Transformers 4.46.3 for this reproducible experiment.")
if tokenizers.__version__ != "0.20.3":
    raise RuntimeError("Use Tokenizers 0.20.3 for this reproducible experiment.")


## 5. Build a 256M starting model and attach vision+text LoRA


In [ ]:
import gc

device = torch.device("cuda:0")
dtype = torch.float16


def load_plain_base():
    options = {
        "cache_dir": str(MODEL_CACHE),
        "torch_dtype": dtype,
        "low_cpu_mem_usage": True,
    }
    try:
        return AutoModelForVision2Seq.from_pretrained(
            MODEL_ID, local_files_only=True, **options
        )
    except OSError:
        print("Local cache incomplete; attempting to download missing base-model files...")
        return AutoModelForVision2Seq.from_pretrained(MODEL_ID, **options)


def load_starting_base():
    base = load_plain_base()
    if START_FROM_PREVIOUS_ADAPTER:
        if PREVIOUS_ADAPTER_DIR is None:
            raise FileNotFoundError(
                "START_FROM_PREVIOUS_ADAPTER=True but the previous adapter was not found."
            )
        previous_model = PeftModel.from_pretrained(base, str(PREVIOUS_ADAPTER_DIR))
        base = previous_model.merge_and_unload()
        print("Merged previous good text-LoRA adapter into the 256M starting base.")
    else:
        print("Starting from the untouched pretrained 256M base.")
    return base


processor_source = (
    str(PREVIOUS_ADAPTER_DIR)
    if START_FROM_PREVIOUS_ADAPTER and PREVIOUS_ADAPTER_DIR is not None
    else MODEL_ID
)
processor = AutoProcessor.from_pretrained(
    processor_source,
    cache_dir=str(MODEL_CACHE),
    local_files_only=True,
)
model = load_starting_base()

ATTENTION_SUFFIXES = ("q_proj", "k_proj", "v_proj", "o_proj", "out_proj")
text_targets, vision_targets = [], []
for name, module in model.named_modules():
    if not isinstance(module, torch.nn.Linear):
        continue
    if not name.endswith(ATTENTION_SUFFIXES):
        continue
    if ".text_model." in name:
        text_targets.append(name)
    elif ".vision_model." in name:
        vision_targets.append(name)

print("Text attention targets:", len(text_targets))
print("Vision attention targets:", len(vision_targets))
if not text_targets or not vision_targets:
    raise RuntimeError(
        "Could not find both text and vision attention projections. "
        "Inspect model.named_modules() before training."
    )

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=sorted(text_targets + vision_targets),
)

model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model = get_peft_model(model, lora_config).to(device)
model.train()
model.print_trainable_parameters()


## 6. Dual-crop dataset and multimodal collator


In [ ]:
import random
from PIL import ImageEnhance
from torch.utils.data import Dataset, DataLoader

INSTRUCTION = (
    "You are shown two views from the same in-cabin driver-camera moment. The first is the full available frame; the second is a close-up of the driver's face. Choose exactly one label. "
    "NORMAL: no clear drowsiness, sustained looking away, or visible phone interaction. "
    "DROWSY: visible sleepiness such as heavy/closed eyes or a head nod. "
    "LOOKING_AWAY: the driver's gaze or head is directed away from the road, without clear visible phone interaction. "
    "PHONE_USAGE: the driver is visibly holding, viewing, or interacting with a phone; do not infer phone use from gaze direction alone. "
    "Use the full frame for phone/hand evidence and the face crop for eye/head cues. If evidence is unclear, choose the best-supported label and do not invent details. "
    "Return exactly one label from NORMAL, DROWSY, LOOKING_AWAY, or PHONE_USAGE, with no explanation or punctuation."
)


class DriverDualCropDataset(Dataset):
    def __init__(self, dataframe, training=False):
        self.dataframe = dataframe.reset_index(drop=True)
        self.training = bool(training)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        full_crop, face_crop, quality_ok, face_detected = create_dual_crops(row["resolved_path"])
        messages = [{
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "image"},
                {"type": "text", "text": INSTRUCTION},
            ],
        }]
        full_image = Image.fromarray(full_crop).convert("RGB")
        face_image = Image.fromarray(face_crop).convert("RGB")
        if (self.training and AUGMENT_LOOKING_AWAY_TRAIN
                and row["label"] == "LOOKING_AWAY" and random.random() < 0.5):
            # Mild, label-preserving photometric augmentation on TRAIN only.
            # Apply the same factors to both views so the paired moment stays consistent.
            brightness = random.uniform(0.90, 1.10)
            contrast = random.uniform(0.90, 1.10)
            full_image = ImageEnhance.Brightness(full_image).enhance(brightness)
            full_image = ImageEnhance.Contrast(full_image).enhance(contrast)
            face_image = ImageEnhance.Brightness(face_image).enhance(brightness)
            face_image = ImageEnhance.Contrast(face_image).enhance(contrast)

        return {
            "full_image": full_image,
            "face_image": face_image,
            "messages": messages,
            "target": row["label"],
            "path": row["resolved_path"],
        }


class DualCropCollator:
    def __init__(self, processor, image_size):
        self.processor = processor
        self.image_size = image_size

    def __call__(self, features):
        images = [[item["full_image"], item["face_image"]] for item in features]
        prompt_texts, full_texts = [], []
        for item in features:
            prompt_messages = item["messages"]
            full_messages = prompt_messages + [{
                "role": "assistant",
                "content": [{"type": "text", "text": item["target"]}],
            }]
            prompt_texts.append(self.processor.apply_chat_template(
                prompt_messages, add_generation_prompt=True, tokenize=False
            ))
            full_texts.append(self.processor.apply_chat_template(
                full_messages, add_generation_prompt=False, tokenize=False
            ))

        kwargs = {
            "return_tensors": "pt",
            "padding": True,
            "truncation": True,
            "size": {"longest_edge": self.image_size},
            "max_image_size": {"longest_edge": self.image_size},
        }
        batch = self.processor(text=full_texts, images=images, **kwargs)
        prompt_batch = self.processor(text=prompt_texts, images=images, **kwargs)
        labels = batch["input_ids"].clone()
        prompt_lengths = prompt_batch["attention_mask"].sum(dim=1)
        for row_index, prompt_length in enumerate(prompt_lengths):
            labels[row_index, : int(prompt_length)] = -100
        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100
        batch["labels"] = labels
        return batch


train_dataset = DriverDualCropDataset(train_df, training=True)
val_dataset = DriverDualCropDataset(val_df, training=False)
test_dataset = DriverDualCropDataset(test_df, training=False)
collator = DualCropCollator(processor, IMAGE_SIZE)

sample_batch = collator([train_dataset[0]])
print("pixel_values:", tuple(sample_batch["pixel_values"].shape))
if sample_batch["pixel_values"].shape[1] != 2:
    raise RuntimeError("Dual-crop validation failed: expected two images per sample.")

# Backward-pass smoke test without an optimizer step, so it does not modify weights.
sample_batch = {
    key: value.to(device) if torch.is_tensor(value) else value
    for key, value in sample_batch.items()
}
smoke_output = model(**sample_batch)
smoke_output.loss.backward()
model.zero_grad(set_to_none=True)
print("Non-updating dual-crop smoke-test loss:", float(smoke_output.loss.detach().cpu()))
del sample_batch, smoke_output
gc.collect()
torch.cuda.empty_cache()


## 7. Generated Macro-F1 model selection and early stopping


In [ ]:
import json
import time
from sklearn.metrics import f1_score
from transformers import TrainerCallback


def parse_prediction(raw_text):
    normalized = str(raw_text).strip().upper()
    exact = normalized in LABELS
    cleaned = normalized.strip(" .,;:!?'\"")
    return (cleaned if cleaned in LABELS else "INVALID"), exact


def predict_rows_batched(model_ref, processor_ref, rows_df, batch_size=8):
    predictions = []
    old_padding = processor_ref.tokenizer.padding_side
    processor_ref.tokenizer.padding_side = "left"
    try:
        for start in range(0, len(rows_df), batch_size):
            rows = rows_df.iloc[start : start + batch_size]
            images, prompts = [], []
            for _, row in rows.iterrows():
                full_crop, face_crop, _, _ = create_dual_crops(row["resolved_path"])
                images.append([
                    Image.fromarray(full_crop).convert("RGB"),
                    Image.fromarray(face_crop).convert("RGB"),
                ])
                messages = [{
                    "role": "user",
                    "content": [
                        {"type": "image"}, {"type": "image"},
                        {"type": "text", "text": INSTRUCTION},
                    ],
                }]
                prompts.append(processor_ref.apply_chat_template(
                    messages, add_generation_prompt=True, tokenize=False
                ))

            inputs = processor_ref(
                text=prompts,
                images=images,
                return_tensors="pt",
                padding=True,
                size={"longest_edge": IMAGE_SIZE},
                max_image_size={"longest_edge": IMAGE_SIZE},
            )
            inputs = {
                key: value.to(next(model_ref.parameters()).device)
                if torch.is_tensor(value) else value
                for key, value in inputs.items()
            }
            with torch.inference_mode():
                generated = model_ref.generate(
                    **inputs, max_new_tokens=12, do_sample=False
                )
            input_length = inputs["input_ids"].shape[-1]
            raw_outputs = processor_ref.batch_decode(
                generated[:, input_length:], skip_special_tokens=True
            )
            predictions.extend(parse_prediction(text)[0] for text in raw_outputs)
    finally:
        processor_ref.tokenizer.padding_side = old_padding
    return predictions


macro_rows = []
for label in LABELS:
    class_rows = val_df[val_df["label"] == label]
    count = min(MACRO_F1_SAMPLES_PER_CLASS, len(class_rows))
    macro_rows.append(class_rows.sample(n=count, random_state=SEED))
macro_f1_df = pd.concat(macro_rows).reset_index(drop=True)
print("Macro-F1 validation subset:", macro_f1_df["label"].value_counts().to_dict())


class MacroF1EarlyStopping(TrainerCallback):
    def __init__(self, processor_ref, check_df, best_dir, patience):
        self.processor_ref = processor_ref
        self.check_df = check_df
        self.best_dir = Path(best_dir)
        self.patience = int(patience)
        self.log_path = OUTPUT_ROOT / "macro_f1_epoch_log.json"
        self.history = []
        self.best_macro_f1 = -1.0
        self.epochs_without_improvement = 0
        if self.log_path.is_file():
            self.history = json.loads(self.log_path.read_text(encoding="utf-8"))
            if self.history:
                scores = [float(row["macro_f1"]) for row in self.history]
                self.best_macro_f1 = max(scores)
                best_index = max(range(len(scores)), key=scores.__getitem__)
                self.epochs_without_improvement = len(scores) - 1 - best_index

    def on_epoch_end(self, args, state, control, **kwargs):
        model_ref = kwargs.get("model")
        if model_ref is None:
            return control
        model_ref.eval()
        started = time.perf_counter()
        try:
            predicted = predict_rows_batched(
                model_ref, self.processor_ref, self.check_df, batch_size=8
            )
            true = self.check_df["label"].astype(str).tolist()
            macro_f1 = f1_score(
                true, predicted, labels=LABELS, average="macro", zero_division=0
            )
            per_class = f1_score(
                true, predicted, labels=LABELS, average=None, zero_division=0
            )
            improved = macro_f1 > self.best_macro_f1
            if improved:
                self.best_macro_f1 = float(macro_f1)
                self.epochs_without_improvement = 0
                self.best_dir.mkdir(parents=True, exist_ok=True)
                model_ref.save_pretrained(str(self.best_dir))
                self.processor_ref.save_pretrained(str(self.best_dir))
            else:
                self.epochs_without_improvement += 1

            record = {
                "epoch": float(state.epoch),
                "macro_f1": float(macro_f1),
                "per_class_f1": dict(zip(LABELS, map(float, per_class))),
                "improved": bool(improved),
                "seconds": float(time.perf_counter() - started),
            }
            self.history.append(record)
            self.log_path.write_text(json.dumps(self.history, indent=2), encoding="utf-8")
            print(
                f"Macro-F1 epoch {state.epoch:.2f}: {macro_f1:.4f} | "
                f"best {self.best_macro_f1:.4f} | per class "
                f"{dict(zip(LABELS, np.round(per_class, 3)))}",
                flush=True,
            )
            if self.epochs_without_improvement >= self.patience:
                print("Macro-F1 patience exhausted; stopping training.", flush=True)
                control.should_training_stop = True
        finally:
            model_ref.train()
        return control


## 8. Terminal-friendly training ETA


In [ ]:
from datetime import datetime, timedelta

PROGRESS_FILE = OUTPUT_ROOT / "training_progress.txt"
PROGRESS_HISTORY = OUTPUT_ROOT / "training_progress_history.log"


def format_duration(seconds):
    hours, remainder = divmod(max(0, int(seconds)), 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


class ETAProgressCallback(TrainerCallback):
    def __init__(self, update_every=25):
        self.update_every = int(update_every)
        self.start_time = None
        self.start_step = 0

    def _write(self, message):
        PROGRESS_FILE.write_text(message + "\n", encoding="utf-8")
        with PROGRESS_HISTORY.open("a", encoding="utf-8") as handle:
            handle.write(message + "\n")
        print(message, flush=True)

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        self.start_step = int(state.global_step)
        self._write(
            f"Training started | step {state.global_step}/{state.max_steps} | "
            f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        )

    def on_step_end(self, args, state, control, **kwargs):
        step = int(state.global_step)
        if step != int(state.max_steps) and step % self.update_every:
            return
        elapsed = time.time() - self.start_time
        completed = max(1, step - self.start_step)
        seconds_per_step = elapsed / completed
        remaining = max(0, int(state.max_steps) - step) * seconds_per_step
        percentage = 100 * step / max(1, int(state.max_steps))
        finish = datetime.now() + timedelta(seconds=remaining)
        self._write(
            f"{percentage:6.2f}% | step {step}/{state.max_steps} | "
            f"elapsed {format_duration(elapsed)} | remaining upper-bound "
            f"{format_duration(remaining)} | ETA {finish.strftime('%Y-%m-%d %H:%M:%S')}"
        )


## 9. Balanced multi-epoch fine-tuning


In [ ]:
from collections import Counter
from torch.utils.data import WeightedRandomSampler
from transformers import TrainingArguments, Trainer, set_seed
from transformers.trainer_utils import get_last_checkpoint

set_seed(SEED)


class ConfigurableSamplerTrainer(Trainer):
    def _get_train_sampler(self):
        if SAMPLER_MODE == "none":
            return super()._get_train_sampler()
        labels = self.train_dataset.dataframe["label"].astype(str).tolist()
        counts = Counter(labels)
        power = 0.5 if SAMPLER_MODE == "sqrt" else 1.0
        weights = torch.tensor(
            [1.0 / (counts[label] ** power) for label in labels], dtype=torch.double
        )
        return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=MAX_EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    fp16=True,
    logging_steps=50,
    disable_tqdm=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=False,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    optim="adamw_torch",
    report_to="none",
    seed=SEED,
)

macro_callback = MacroF1EarlyStopping(
    processor_ref=processor,
    check_df=macro_f1_df,
    best_dir=BEST_ADAPTER_DIR,
    patience=EARLY_STOPPING_PATIENCE,
)

trainer = ConfigurableSamplerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    callbacks=[ETAProgressCallback(update_every=25), macro_callback],
)

last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))
print("Resume checkpoint:", last_checkpoint)
print("Starting balanced dual-crop vision+text LoRA training...")
print("Progress file:", PROGRESS_FILE)

train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
print("Training ended:", train_result)
trainer.save_model(str(LAST_ADAPTER_DIR))
processor.save_pretrained(str(LAST_ADAPTER_DIR))
print("Last adapter:", LAST_ADAPTER_DIR)
print("Macro-F1-best adapter:", BEST_ADAPTER_DIR)


## 10. Reload the Macro-F1-best model


In [ ]:
if not (BEST_ADAPTER_DIR / "adapter_model.safetensors").is_file():
    raise FileNotFoundError("Macro-F1-best adapter was not saved.")

del trainer, model
gc.collect()
torch.cuda.empty_cache()

eval_processor = AutoProcessor.from_pretrained(str(BEST_ADAPTER_DIR), local_files_only=True)
eval_base = load_starting_base()
eval_model = PeftModel.from_pretrained(eval_base, str(BEST_ADAPTER_DIR)).to(device)
eval_model.eval()
eval_model.config.use_cache = True
print("Loaded Macro-F1-best model:", BEST_ADAPTER_DIR)


## 11. Full, resumable Phase 2 test evaluation


In [ ]:
from tqdm.auto import tqdm

PREDICTIONS_CSV = OUTPUT_ROOT / "phase2_256m_all_techniques_test_predictions.csv"
SAVE_EVERY = 250


def predict_test_row(row):
    full_crop, face_crop, _, _ = create_dual_crops(row["resolved_path"])
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"}, {"type": "image"},
            {"type": "text", "text": INSTRUCTION},
        ],
    }]
    prompt = eval_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    inputs = eval_processor(
        text=prompt,
        images=[[Image.fromarray(full_crop), Image.fromarray(face_crop)]],
        return_tensors="pt",
        size={"longest_edge": IMAGE_SIZE},
        max_image_size={"longest_edge": IMAGE_SIZE},
    )
    inputs = {
        key: value.to(device) if torch.is_tensor(value) else value
        for key, value in inputs.items()
    }
    started = time.perf_counter()
    with torch.inference_mode():
        generated = eval_model.generate(
            **inputs, max_new_tokens=12, do_sample=False
        )
    latency_ms = (time.perf_counter() - started) * 1000
    input_length = inputs["input_ids"].shape[-1]
    raw = eval_processor.batch_decode(
        generated[:, input_length:], skip_special_tokens=True
    )[0].strip()
    prediction, exact = parse_prediction(raw)
    return raw, prediction, exact, latency_ms


if PREDICTIONS_CSV.is_file():
    records = pd.read_csv(PREDICTIONS_CSV).to_dict("records")
    finished = {int(row["row_index"]) for row in records}
    print("Resuming after", len(finished), "test rows.")
else:
    records, finished = [], set()

torch.cuda.reset_peak_memory_stats()
remaining = [index for index in range(len(test_df)) if index not in finished]
for completed, index in enumerate(
    tqdm(remaining, desc="Evaluating full Phase 2 test set", mininterval=2), start=1
):
    row = test_df.iloc[index]
    raw, prediction, exact, latency_ms = predict_test_row(row)
    records.append({
        "row_index": index,
        "frame_path": row["frame_path"],
        "true_label": row["label"],
        "raw_output": raw,
        "prediction": prediction,
        "exact_label_valid": exact,
        "latency_ms": latency_ms,
    })
    if completed % SAVE_EVERY == 0:
        pd.DataFrame(records).sort_values("row_index").to_csv(PREDICTIONS_CSV, index=False)

predictions_df = (
    pd.DataFrame(records)
    .sort_values("row_index")
    .drop_duplicates("row_index", keep="last")
)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)
print("Saved test rows:", len(predictions_df), "/", len(test_df))
print("Predictions:", PREDICTIONS_CSV)


## 12. Final metrics


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support

if len(predictions_df) != len(test_df):
    raise RuntimeError("Full test evaluation is incomplete.")

y_true = predictions_df["true_label"].astype(str).to_numpy()
y_pred = predictions_df["prediction"].astype(str).to_numpy()
latencies = predictions_df["latency_ms"].astype(float).to_numpy()

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=LABELS, zero_division=0
)
unsafe_true = np.isin(y_true, ["DROWSY", "LOOKING_AWAY", "PHONE_USAGE"])
normal_true = y_true == "NORMAL"
unsafe_pred = np.isin(y_pred, ["DROWSY", "LOOKING_AWAY", "PHONE_USAGE"])

unsafe_to_safe_miss = np.mean(np.isin(y_pred[unsafe_true], ["NORMAL", "INVALID"]))
unsafe_class_error = np.mean(y_pred[unsafe_true] != y_true[unsafe_true])
false_alarm = np.mean(unsafe_pred[normal_true])
exact_validity = predictions_df["exact_label_valid"].astype(bool).mean()
parser_failure = np.mean(y_pred == "INVALID")
matrix_labels = LABELS + ["INVALID"]
cm = confusion_matrix(y_true, y_pred, labels=matrix_labels)

print("\n========== 256M ALL-TECHNIQUES PHASE 2 TEST ==========")
print("Images:", len(predictions_df))
print("Accuracy:", round(accuracy_score(y_true, y_pred) * 100, 2), "%")
print("Balanced accuracy:", round(recall.mean() * 100, 2), "%")
print("Macro-F1:", round(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0) * 100, 2), "%")
print("Exact-label validity:", round(exact_validity * 100, 2), "%")
print("Parser failure:", round(parser_failure * 100, 2), "%")
print("\nPer-class results:")
for label, p, r, score, count in zip(LABELS, precision, recall, f1, support):
    print(f"{label}: precision={p:.3f}, recall={r:.3f}, F1={score:.3f}, samples={count}")
print("\nConfusion matrix:")
print(pd.DataFrame(cm, index=matrix_labels, columns=matrix_labels))
print("Unsafe-to-safe miss rate:", round(unsafe_to_safe_miss * 100, 2), "%")
print("Unsafe-class error rate:", round(unsafe_class_error * 100, 2), "%")
print("False-alarm rate:", round(false_alarm * 100, 2), "%")
print("Mean latency (ms):", round(float(np.mean(latencies)), 2))
print("P95 latency (ms):", round(float(np.percentile(latencies, 95)), 2))
print("Peak GPU memory (MB):", round(torch.cuda.max_memory_allocated() / 1024**2, 2))


# Save a review table for post-evaluation error analysis. Do not tune model
# settings on this test-set table; use validation data for experiment selection.
ERROR_REVIEW_CSV = OUTPUT_ROOT / "test_misclassification_review.csv"
error_rows = predictions_df[predictions_df["true_label"] != predictions_df["prediction"]].copy()
error_rows.to_csv(ERROR_REVIEW_CSV, index=False)
print("Saved test misclassification review (reporting only):", ERROR_REVIEW_CSV)
print("Misclassifications by true/predicted label:")
print(error_rows.groupby(["true_label", "prediction"]).size().sort_values(ascending=False).head(20))


## 13. Optional merged FP32 deployment export


In [ ]:
CREATE_MERGED_FP32 = False

if CREATE_MERGED_FP32:
    MERGED_DIR = OUTPUT_ROOT / "merged_model_256m_all_techniques_fp32"
    merged_model = eval_model.merge_and_unload().float()
    merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
    eval_processor.save_pretrained(str(MERGED_DIR))
    print("Merged FP32 model:", MERGED_DIR)
else:
    print("Merge skipped until the new test results are accepted.")
